# Layer 2: End-user demand
Reporting probit → inverse probability weights → PPML → missing sell-through predictions → aggregation. Read README_LAYER2.md for assumptions and fixed-effect choices.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from layer2 import run

DATA = Path("data")
LAYER1 = Path("results/layer1")
OUTPUT = Path("results/layer2")
assert (LAYER1 / "layer2_input.csv").exists(), "Run Layer 1 first."
panel = pd.read_csv(LAYER1 / "layer2_input.csv")
display(panel.head())

## Estimate and save
The function fits reporting probabilities on the full panel, then fits PPML on observed sell-through using raw 1/pi weights. Missing outcomes are never treated as zeros. evaluate=False keeps all hidden truth out of this run.

In [ ]:
enduser, coefficients, diagnostics = run(DATA, LAYER1, OUTPUT, evaluate=False)
display(coefficients.head(4))
print(json.dumps(diagnostics, indent=2))

## Inspect reporting coverage and IPW
No automatic weight trimming is applied. Very small probabilities require investigating overlap and defining any trimming rule explicitly.

In [ ]:
display(enduser.groupby("reporting_indicator").agg(
    observations=("reseller_id", "size"), observed_outcomes=("sellthrough_quantity", "count"),
    mean_probability=("pi_hat", "mean")
))
display(enduser.loc[enduser.reporting_indicator.eq(1), "ipw"].describe(percentiles=[.01,.05,.5,.95,.99]))

## Complete and aggregate demand
Use observed sales for reporters and predicted conditional means for nonreporters. The all-model prediction is also saved separately.

In [ ]:
display(enduser[["reseller_id", "sku", "market_period_id", "reporting_indicator",
    "sellthrough_quantity", "R_hat", "pi_hat", "predicted_enduser_mean", "completed_enduser_quantity"]].head(12))
product_totals = pd.read_csv(OUTPUT / "product_total_demand.csv")
display(product_totals)

## Optional simulation evaluation
The package includes evaluation from the completed command-line run: core_parameter_recovery.csv and evaluation_metrics.json. Evaluation uses truth only after fitting. To recompute it, run the following optional cell. The oracle uses true reseller demand solely to diagnose the generated-regressor issue.

In [ ]:
# Optional: uncomment to refit and evaluate.
# enduser, coefficients, diagnostics = run(DATA, LAYER1, OUTPUT, evaluate=True)
if (OUTPUT / "core_parameter_recovery.csv").exists():
    display(pd.read_csv(OUTPUT / "core_parameter_recovery.csv"))
    print((OUTPUT / "evaluation_metrics.json").read_text())